# Essay Figures

This notebook regenerates the two essay figures:

1. `latency_breakdown_cascade.png`
2. `failure_mode_layer_matrix.png`

It is dependency-light and uses only Pillow so it can run in this repo without installing matplotlib. The numeric values are drawn from the reported experiment summaries and kept explicit here for reviewer traceability.


In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

OUT = Path('essay') if Path('essay').exists() else Path('.')

latency_stages = [
    ('Round 01 raw', 16.5549, '30 seed / 28 completed'),
    ('Round 02 controlled', 6.2214, '200 gold-only'),
    ('V3.1 prune + tags', 3.3121, '300 Qwen rows'),
    ('V3.3 postprocessed', 2.4908, '300 Qwen rows'),
]
v33_breakdown = [
    ('Prompt encoding / TTFT', 1.1058),
    ('Prose generation', 1.3850),
    ('Polish + tag injection', 0.005),
]
failure_rows = [
    ('Deterministic refusal lane', 'G002 refusal failures', '8 failures', '0 fail/warn'),
    ('Deterministic source/rights lane', 'G005/G006/G007 rights-source errors', '47 + 12 + 3 findings', '0 fail/warn'),
    ('Evidence tag injection', 'G101 field visibility', '146 warnings; 61 fails', '0 fail/warn'),
    ('Prose polisher', 'Overconfidence / unsupported first claims', '3/191 overconfidence', '0 overconfidence'),
    ('Evidence pruning + compact packets', 'Qwen P95 latency', '~8.0s P95', '4.19s P95'),
]

font_candidates = [
    '/System/Library/Fonts/Supplemental/Arial.ttf',
    '/System/Library/Fonts/Supplemental/Helvetica.ttf',
    '/System/Library/Fonts/Supplemental/Verdana.ttf',
]
def font(size, bold=False):
    candidates = []
    if bold:
        candidates += [
            '/System/Library/Fonts/Supplemental/Arial Bold.ttf',
            '/System/Library/Fonts/Supplemental/Helvetica Bold.ttf',
        ]
    candidates += font_candidates
    for p in candidates:
        try:
            return ImageFont.truetype(p, size)
        except Exception:
            pass
    return ImageFont.load_default()

F_TITLE=font(42,True); F_SUB=font(22); F_BODY=font(20); F_BODY_B=font(20,True); F_SMALL=font(16); F_TINY=font(14)
COL={'ink':(29,36,45),'muted':(88,101,116),'grid':(218,224,232),'blue':(38,105,186),'teal':(30,150,140),'green':(47,153,95),'orange':(232,139,45),'red':(214,76,76),'purple':(121,92,180),'light_green':(226,244,235),'white':(255,255,255),'bg':(248,250,252)}
def rounded(d,xy,r,fill,outline=None,width=1):
    d.rounded_rectangle(xy,radius=r,fill=fill,outline=outline,width=width)
def wrap(d,text,max_width,font_obj):
    words=str(text).split(); lines=[]; line=''
    for w in words:
        cand=(line+' '+w).strip()
        if d.textbbox((0,0),cand,font=font_obj)[2] <= max_width:
            line=cand
        else:
            if line: lines.append(line)
            line=w
    if line: lines.append(line)
    return lines

def draw_latency():
    W,H=1700,1120
    img=Image.new('RGB',(W,H),COL['bg']); d=ImageDraw.Draw(img)
    d.text((60,45),'Latency Breakdown Cascade',font=F_TITLE,fill=COL['ink'])
    d.text((62,100),'Stage-level latency reduction and V3.3 component decomposition',font=F_SUB,fill=COL['muted'])
    left,top=90,210; bar_w_max=890; bar_h=58; row_gap=126; maxv=latency_stages[0][1]
    colors=[COL['red'],COL['orange'],COL['teal'],COL['green']]
    for i,(name,val,note) in enumerate(latency_stages):
        y=top+i*row_gap
        d.text((left,y-34),name,font=F_BODY_B,fill=COL['ink'])
        d.text((left+250,y-31),note,font=F_SMALL,fill=COL['muted'])
        bw=int(bar_w_max*val/maxv)
        rounded(d,(left,y,left+bw,y+bar_h),12,colors[i])
        d.text((left+bw+18,y+15),f'{val:.2f}s',font=F_BODY_B,fill=COL['ink'])
    rounded(d,(90,740,1035,810),16,COL['white'],outline=COL['grid'])
    d.text((120,760),'Overall Qwen-row reduction: 16.55s -> 2.49s (-85.0%)',font=F_BODY_B,fill=COL['ink'])
    d.text((120,790),'Stage benchmarks mix sample sizes; paired Round 03 comparisons are reported separately.',font=F_SMALL,fill=COL['muted'])
    px,py,pw,ph=1110,185,520,510
    rounded(d,(px,py,px+pw,py+ph),22,COL['white'],outline=COL['grid'],width=2)
    d.text((px+30,py+30),'V3.3 Qwen average: 2.49s',font=F_BODY_B,fill=COL['ink'])
    d.text((px+30,py+62),'What the final latency contains',font=F_SMALL,fill=COL['muted'])
    comps=[('Prompt encoding / TTFT',1.1058,COL['blue']),('Prose generation',1.3850,COL['teal']),('Polish + tag injection',0.005,COL['green'])]
    stack_x,stack_y=px+45,py+135; stack_w,stack_h=pw-90,72; total=sum(v for _,v,_ in comps); x=stack_x
    for label,val,c in comps:
        sw=max(4,int(stack_w*val/total)); d.rectangle((x,stack_y,x+sw,stack_y+stack_h),fill=c); x+=sw
    d.rectangle((stack_x,stack_y,stack_x+stack_w,stack_y+stack_h),outline=COL['ink'],width=2)
    ly=stack_y+112
    for label,val,c in comps:
        d.rectangle((px+45,ly+4,px+65,ly+24),fill=c)
        shown='<5ms' if val<0.01 else f'{val:.2f}s'
        d.text((px+78,ly),f'{label}: {shown}',font=F_SMALL,fill=COL['ink'])
        ly+=46
    rounded(d,(px+35,py+405,px+pw-35,py+460),14,COL['light_green'])
    d.text((px+55,py+422),'System overhead is effectively negligible.',font=F_SMALL,fill=COL['ink'])
    d.text((60,H-58),'Sources: Round 01, Round 02 gold-only, V3.1 300, and V3.3 300 reports in /reports.',font=F_TINY,fill=COL['muted'])
    img.save(OUT/'latency_breakdown_cascade.png')

def draw_failure_matrix():
    W,H=1800,1220
    img=Image.new('RGB',(W,H),COL['bg']); d=ImageDraw.Draw(img)
    d.text((60,45),'Decoupled Layers vs Failure Modes',font=F_TITLE,fill=COL['ink'])
    d.text((62,100),'Each layer prevents a specific observed collapse mode',font=F_SUB,fill=COL['muted'])
    color_map=[COL['red'],COL['orange'],COL['purple'],COL['teal'],COL['blue']]
    x0,y0=65,180; col_w=[420,430,390,320]; headers=['Decoupling layer','Protected metric','Without layer','With V3.3']; row_h=160
    x=x0
    for i,hdr in enumerate(headers):
        rounded(d,(x,y0,x+col_w[i]-10,y0+70),10,COL['ink'])
        d.text((x+18,y0+22),hdr,font=F_BODY_B,fill=COL['white']); x+=col_w[i]
    for r,(layer,metric,without,withv) in enumerate(failure_rows):
        y=y0+86+r*row_h; x=x0; fills=[COL['white'],COL['white'],(255,244,244),COL['light_green']]; vals=[layer,metric,without,withv]
        for c_idx,val in enumerate(vals):
            rounded(d,(x,y,x+col_w[c_idx]-10,y+row_h-18),12,fills[c_idx],outline=COL['grid'])
            if c_idx==0: d.rectangle((x,y,x+10,y+row_h-18),fill=color_map[r])
            text_x=x+24; max_w=col_w[c_idx]-56; lines=wrap(d,val,max_w,F_BODY_B if c_idx in (0,2,3) else F_BODY); ty=y+32
            for line in lines[:4]:
                d.text((text_x,ty),line,font=F_BODY_B if c_idx in (0,2,3) else F_BODY,fill=COL['ink']); ty+=30
            x+=col_w[c_idx]
    note_y=y0+86+len(failure_rows)*row_h+18
    rounded(d,(65,note_y,1705,note_y+95),18,COL['white'],outline=COL['grid'])
    d.text((95,note_y+20),'Reviewer-facing interpretation:',font=F_BODY_B,fill=COL['ink'])
    d.text((95,note_y+52),'The final score is less important than the layer-removal evidence: each reliability layer corresponds to an observed failure class.',font=F_BODY,fill=COL['muted'])
    d.text((60,H-55),'Sources: FAILURE_MODE_ANALYSIS_V33.md; WEBLLM_ROUND_01.md; ROUND_02_200_RUNTIME_RESULTS.md; V3.1/V3.2/V3.3 Round 03 reports.',font=F_TINY,fill=COL['muted'])
    img.save(OUT/'failure_mode_layer_matrix.png')

draw_latency()
draw_failure_matrix()
print('Wrote figure PNGs to', OUT)
